# Nine PEAS agents, live

From [jmurray10/agent_design_peas](https://github.com/jmurray10/agent_design_peas). Nine agents run through one
`ConfigDrivenAgent` class that contains no agent-specific code. An agent is a
directory of YAML, prompts and JSON Schema; adding one adds a directory, not a
branch.

They cover all five classical architectures:

| architecture | agents |
|---|---|
| simple reflex | claims-intake, data-contract, safety-signal, uptime-triage |
| model-based reflex | aml-alert, support-bot |
| goal-based | trial-scheduler (a CSP: the solver proves a request unsatisfiable) |
| utility-based | claim-reserve (both errors cost money, asymmetrically) |
| learning | triage-tuner (has an explore action; its critic is arithmetic) |

**Two ways to run this.** Without a key it replays what a real model actually
returned to each prompt, recorded in the repository with the model name and date.
With your own Anthropic key it calls the model live. There is no third mode: a
prompt with no recording raises rather than inventing an answer.

## 1. Get the code


In [ ]:
!git clone --depth 1 https://github.com/jmurray10/agent_design_peas.git peas 2>/dev/null || echo 'already cloned'
%cd peas
!pip install -q pyyaml jsonschema anthropic

## 2. Optional: a live model

Skip this cell to watch the replay instead. Nothing below changes either way,
which is the point of the shim.

Put your key in Colab's Secrets panel (the key icon in the left sidebar) as
`ANTHROPIC_API_KEY` rather than pasting it into a cell.

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    key = None

if key:
    os.environ['ANTHROPIC_API_KEY'] = key
    os.environ['LLM_PROVIDER'] = 'anthropic'
    print('Live. Every call below reaches a real model.')
else:
    print('No key found. Replaying recorded real responses instead.')

## 3. Every agent, driven by its own eval cases

`demo.py` discovers the agent directories, runs each against the cases in its own
`eval/` folder, hands each one a percept no sensor schema accepts to show the
refusal, and finishes by checking that `runtime.py` mentions none of the ninety
names belonging to a specific agent.

In [ ]:
!python 00-config-runtime/demo.py

## 4. The claim a single-percept suite cannot test

Every case above judges one percept on its own, which is what the HTTP contract
promises a caller and is structurally unable to test whether an agent *learns*.
This harness runs the same percept twice -- once cold, once after a preamble it
should have learned from -- and passes only when the two answers differ in the
declared direction.

In [ ]:
!python 00-config-runtime/sequence_eval.py

## 5. One agent, one percept

The percept below ships with a recording, so it works with or without a key.

**Edit it and it will stop working without one**, and that is deliberate:
recordings are keyed by the content of the prompt, so a changed percept misses
the recording and raises instead of replaying an answer to a question nobody
asked. Add your key in the cell above to explore freely.

A percept no sensor schema accepts is refused before a model is called at all,
which is the 422 in the HTTP version of this agent.

In [ ]:
import sys, pathlib
sys.path.insert(0, '00-config-runtime')
from runtime import ConfigDrivenAgent, PerceptRejected

agent = ConfigDrivenAgent(pathlib.Path('00-config-runtime/agents/claim-reserve'))

percept = {
    'claim_id': 'RS-9001',
    'loss_type': 'property_fire',
    'description': 'Restaurant kitchen fire, extinguished by the sprinkler system. '
                   'Kitchen gutted, dining room smoke-damaged, closed for repairs.',
    'liability_clear': True,
    'initial_estimate': 0.0,
    'injury_severity': 'none',
    'policy_limit': 2000000.0,
    'assessment_scheduled': True,
}

try:
    result = agent.run(percept)
    print('action :', result['action'])
    print('reason :', result['args'].get('reason'))
    print('range  :', result['args'].get('expected_range'))
except PerceptRejected as refusal:
    print('refused before any model call:', refusal)
except LookupError:
    print('No recording exists for this exact percept, and no key is set,')
    print('so there is nothing to replay. That failure is the design: the')
    print('alternative is inventing an answer, and a demo that invents')
    print('answers stops meaning anything.')
    print()
    print('Add a key in the cell above, or restore the percept this')
    print('notebook ships with.')


## What to notice

The eval standings are untuned. Where a model disagreed with an expectation, the
expectation was left alone unless reading the disagreement showed the case was
wrong -- and several were. The repository's history records which, and why, one
commit at a time.

`uptime-triage` is the honest case for the cheapest model here: the decision runs
on every alert of every day, so a frontier model on each one is not a design
anyone can afford.

If you ran this without a key, you watched a replay of real model output rather
than a simulation of one. That is not the same as running your own, and the
repository says so in its README rather than pretending otherwise.